In [2]:
import numpy as np
import pickle


# ==================================================
# 0. Global setting
# ==================================================

# Fix random seed for reproducible results
RANDOM_SEED = 559
np.random.seed(RANDOM_SEED)


# ==================================================
# 1. Dataset settings
# ==================================================

# Run the same SVM model on two different feature sets
datasets = [
    {
        "name": "PCA Features",
        "data_path": "../data/preprocessed_data_outlier_pca.pkl",
        "threshold": -0.0655546589142745
    },
    {
        "name": "VAE-Enhanced Features",
        "data_path": "../data/preprocessed_data_vae_enhanced.pkl",
        "threshold": -0.0655546589142745
    }
]


# ==================================================
# 2. Metric functions
# ==================================================

def confusion_matrix_manual(y_true, y_pred):
    # In this project, -1 means Pass and 1 means Fail
    tn = np.sum((y_true == -1) & (y_pred == -1))
    fp = np.sum((y_true == -1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == -1))
    tp = np.sum((y_true == 1) & (y_pred == 1))

    return np.array([[tn, fp],
                     [fn, tp]])


def compute_metrics(y_true, y_pred):
    # Compute confusion matrix first
    cm = confusion_matrix_manual(y_true, y_pred)

    tn = cm[0, 0]
    fp = cm[0, 1]
    fn = cm[1, 0]
    tp = cm[1, 1]

    # Basic evaluation metrics
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    # F1 balances precision and recall
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    # F2 gives more weight to recall
    f2 = (
        5 * precision * recall / (4 * precision + recall)
        if 4 * precision + recall > 0
        else 0.0
    )

    return accuracy, precision, recall, f1, f2, cm


def print_result(title, y_true, y_pred):
    # Print all metrics in the same format
    accuracy, precision, recall, f1, f2, cm = compute_metrics(y_true, y_pred)

    print("\n" + title)
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1-score:", f1)
    print("F2-score:", f2)
    print("Confusion Matrix:")
    print(cm)


# ==================================================
# 3. Manual Linear SVM
# ==================================================

class ManualLinearSVM:
    def __init__(
        self,
        C=0.05,
        class_weight={-1: 1.0, 1: 5.0},
        learning_rate=5e-05,
        epochs=5000,
        batch_size=128,
        random_seed=559
    ):
        # C controls the penalty for margin violations
        self.C = C

        # Use a larger weight for the Fail class due to class imbalance
        self.class_weight = class_weight

        # Training parameters
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.random_seed = random_seed

        # Model parameters
        self.w = None
        self.b = 0.0

    def _get_sample_weights(self, y):
        # Start with weight 1 for all samples
        weights = np.ones(len(y))

        # Apply class weights manually
        if isinstance(self.class_weight, dict):
            weights[y == -1] = self.class_weight.get(-1, 1.0)
            weights[y == 1] = self.class_weight.get(1, 1.0)

        return weights

    def fit(self, X, y):
        # Reset seed before training
        np.random.seed(self.random_seed)

        n_samples, n_features = X.shape

        # Initialize linear model parameters
        self.w = np.zeros(n_features)
        self.b = 0.0

        sample_weights = self._get_sample_weights(y)

        for epoch in range(self.epochs):
            # Shuffle data at each epoch
            indices = np.random.permutation(n_samples)

            X_shuffled = X[indices]
            y_shuffled = y[indices]
            weight_shuffled = sample_weights[indices]

            # Mini-batch gradient descent
            for start in range(0, n_samples, self.batch_size):
                end = start + self.batch_size

                X_batch = X_shuffled[start:end]
                y_batch = y_shuffled[start:end]
                weight_batch = weight_shuffled[start:end]

                # Compute linear scores
                scores = X_batch @ self.w + self.b

                # Compute SVM margins
                margins = y_batch * scores

                # Hinge loss only applies when margin < 1
                active = margins < 1

                # L2 regularization gradient
                grad_w = self.w.copy()
                grad_b = 0.0

                if np.any(active):
                    X_active = X_batch[active]
                    y_active = y_batch[active]
                    weight_active = weight_batch[active]

                    # Weighted hinge loss gradient
                    grad_w -= self.C * np.mean(
                        weight_active[:, None] * y_active[:, None] * X_active,
                        axis=0
                    )

                    grad_b -= self.C * np.mean(weight_active * y_active)

                # Update weights and bias
                self.w -= self.learning_rate * grad_w
                self.b -= self.learning_rate * grad_b

    def decision_function(self, X):
        # Return raw model scores
        return X @ self.w + self.b

    def predict(self, X, threshold=0.0):
        # Convert scores into class labels using the selected threshold
        scores = self.decision_function(X)
        return np.where(scores >= threshold, 1, -1)


# ==================================================
# 4. Run final SVM on both datasets
# ==================================================

all_results = []

for dataset in datasets:
    print("\n" + "=" * 70)
    print("Dataset:", dataset["name"])
    print("=" * 70)

    # Load one preprocessed dataset
    with open(dataset["data_path"], "rb") as f:
        preprocessed_data = pickle.load(f)

    # Convert data to NumPy arrays
    X_train = np.array(preprocessed_data["X_train"], dtype=np.float64)
    X_val   = np.array(preprocessed_data["X_val"], dtype=np.float64)
    X_test  = np.array(preprocessed_data["X_test"], dtype=np.float64)

    y_train = np.array(preprocessed_data["y_train"], dtype=int)
    y_val   = np.array(preprocessed_data["y_val"], dtype=int)
    y_test  = np.array(preprocessed_data["y_test"], dtype=int)

    # Print data shapes and class distribution
    print(f"X_train: {X_train.shape}")
    print(f"X_val:   {X_val.shape}")
    print(f"X_test:  {X_test.shape}")
    print(f"y_train distribution: -1={np.sum(y_train == -1)}, +1={np.sum(y_train == 1)}")
    print(f"y_val distribution:   -1={np.sum(y_val == -1)}, +1={np.sum(y_val == 1)}")
    print(f"y_test distribution:  -1={np.sum(y_test == -1)}, +1={np.sum(y_test == 1)}")

    # Train the final manual SVM model
    final_svm = ManualLinearSVM(
        C=0.05,
        class_weight={-1: 1.0, 1: 5.0},
        learning_rate=5e-05,
        epochs=5000,
        batch_size=128,
        random_seed=RANDOM_SEED
    )

    final_svm.fit(X_train, y_train)

    # Use the selected threshold for this dataset
    selected_threshold = dataset["threshold"]

    y_train_pred = final_svm.predict(X_train, threshold=selected_threshold)
    y_val_pred = final_svm.predict(X_val, threshold=selected_threshold)
    y_test_pred = final_svm.predict(X_test, threshold=selected_threshold)

    # Print final model settings
    print("\nFinal Configuration")
    print("C:", 0.05)
    print("class_weight:", {-1: 1.0, 1: 5.0})
    print("learning_rate:", 5e-05)
    print("epochs:", 5000)
    print("batch_size:", 128)
    print("threshold:", selected_threshold)

    # Print train, validation, and test results
    print_result("Training Result", y_train, y_train_pred)
    print_result("Validation Result", y_val, y_val_pred)
    print_result("Final Test Result", y_test, y_test_pred)

    # Compute train and validation error
    train_metrics = compute_metrics(y_train, y_train_pred)
    val_metrics = compute_metrics(y_val, y_val_pred)
    test_metrics = compute_metrics(y_test, y_test_pred)

    train_error = 1 - train_metrics[0]
    val_error = 1 - val_metrics[0]

    print("\nTraining vs Validation Error")
    print("Training Error:", train_error)
    print("Validation Error:", val_error)

    # Store test results for final comparison
    all_results.append({
        "dataset": dataset["name"],
        "accuracy": test_metrics[0],
        "precision": test_metrics[1],
        "recall": test_metrics[2],
        "f1": test_metrics[3],
        "f2": test_metrics[4],
        "confusion_matrix": test_metrics[5]
    })


# ==================================================
# 5. Summary
# ==================================================

print("\n" + "=" * 70)
print("Summary")
print("=" * 70)

# Compare the final test results from both feature sets
for result in all_results:
    print("\nDataset:", result["dataset"])
    print("Accuracy:", result["accuracy"])
    print("Precision:", result["precision"])
    print("Recall:", result["recall"])
    print("F1-score:", result["f1"])
    print("F2-score:", result["f2"])
    print("Confusion Matrix:")
    print(result["confusion_matrix"])


Dataset: PCA Features
X_train: (1096, 20)
X_val:   (235, 20)
X_test:  (236, 20)
y_train distribution: -1=1023, +1=73
y_val distribution:   -1=220, +1=15
y_test distribution:  -1=220, +1=16

Final Configuration
C: 0.05
class_weight: {-1: 1.0, 1: 5.0}
learning_rate: 5e-05
epochs: 5000
batch_size: 128
threshold: -0.0655546589142745

Training Result
Accuracy: 0.635036496350365
Precision: 0.12064965197215777
Recall: 0.7123287671232876
F1-score: 0.20634920634920634
F2-score: 0.359612724757953
Confusion Matrix:
[[644 379]
 [ 21  52]]

Validation Result
Accuracy: 0.625531914893617
Precision: 0.12371134020618557
Recall: 0.8
F1-score: 0.2142857142857143
F2-score: 0.38216560509554137
Confusion Matrix:
[[135  85]
 [  3  12]]

Final Test Result
Accuracy: 0.5635593220338984
Precision: 0.10810810810810811
Recall: 0.75
F1-score: 0.1889763779527559
F2-score: 0.34285714285714286
Confusion Matrix:
[[121  99]
 [  4  12]]

Training vs Validation Error
Training Error: 0.36496350364963503
Validation Error: 